# From the current ~40/60 to 50+: where the points go

Existing development CSVs only; no fitting, new inference, images or locked-result CSVs. See `PROTOCOL_codex.md`. Scores and wrong answers stay together. Counterfactual bounds and truth-picked combinations are diagnostic accounting, not deployable improvements. The current comparison configuration is XoFTR from 800 m (experimental switch), not the default settings file.

Scoring: match within 1/2/4/8 sigma earns 10/6/3/0, beyond 8 sigma -20; skip 0. Overall score is 6 times mean points, with skips included.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

root = Path.cwd().resolve()
while not (root / 'REPO_MAP.md').exists():
    root = root.parent
folder = root / 'tests/Score_Path_20261118_codex'
results = folder / 'results'
sys.path.insert(0, str(folder / 'changed_src'))
from score_budget_codex import load_development, summarize_variants, grouped_budget, calculate_bounds, summarize_stage_diagnostics, file_digest
source_folder = root / 'tests/Test_Iteration_12/results'
metadata_path = source_folder / 'native_fine/metadata_dev332.csv'
split_path = source_folder / 'highalt_split.csv'
metadata = pd.read_csv(metadata_path)
split = pd.read_csv(split_path)
band_edges = [100,300,500,800,1000,1300,1600,2100]
band_labels = ['100–300','300–500','500–800','800–1000','1000–1300','1300–1600','1600–2017']
variant_files = {'LoFTR baseline':'details_base.csv', 'LoFTR native':'details_native800.csv', 'XoFTR 800':'details_xoftr800.csv', 'XoFTR native':'details_xoftr800_native800.csv'}
frames = {}
source_hashes = {str(metadata_path.relative_to(root)):file_digest(metadata_path), str(split_path.relative_to(root)):file_digest(split_path)}
for name, filename in variant_files.items():
    path = source_folder / 'native_fine' / filename
    frames[name] = load_development(path, metadata, split, band_edges, band_labels)
    source_hashes[str(path.relative_to(root))] = file_digest(path)
assert len(metadata) == 332
current = frames['XoFTR 800']
print('Loaded four development runs, each with all 332 rows. No test_details files read.')


In [ ]:
summary = summarize_variants(frames)
summary.to_csv(results / 'variant_summary_codex.csv', index=False)
display(summary[['variant','score60_within_group','answered','wrong_beyond8sigma','precision_headroom60','skip_headroom60','mean_seconds','max_seconds']].round(3))
bands = grouped_budget(current, 'band')
levels = grouped_budget(current, 'level')
bands.to_csv(results / 'current_band_budget_codex.csv', index=False)
levels.to_csv(results / 'current_level_budget_codex.csv', index=False)
display(bands[['band','n','answered','full_points','six_points','three_points','points_per_photo','wrong_beyond8sigma','precision_headroom60','skip_headroom60']].round(3))
display(levels[['level','n','points_per_photo','answered','wrong_beyond8sigma','precision_headroom60','skip_headroom60']].round(3))
for name, frame in frames.items():
    below = frame[frame.alt < 800].set_index('filename').points
    baseline = frames['LoFTR baseline'].query('alt < 800').set_index('filename').points
    assert below.sort_index().equals(baseline.sort_index())
print('Below 800 m, stored points are identical across all four variants.')


In [ ]:
bounds = calculate_bounds(frames, 'XoFTR 800', 50.0)
(results / 'counterfactual_bounds_codex.json').write_text(json.dumps(bounds, indent=2))
print(json.dumps(bounds, indent=2))
assert abs(summary.loc[summary.variant.eq('XoFTR 800'),'score60_within_group'].iloc[0] + bands.precision_headroom60.sum() + bands.skip_headroom60.sum() - 60) < 1e-9
print('Bounds are mutually different hypothetical interventions; do not add their scores.')
print('Truth-picked best of four uses true outcome labels and cannot be deployed.')
print('Only these four completed configurations are bounded; new geometry/matching methods are not.')


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios':[1.6,1]})
position = np.arange(len(bands))
earned = bands.points_per_photo.to_numpy()
precision = bands.precision_lost_raw.to_numpy() / bands.n.to_numpy()
skipped = bands.skip_lost_raw.to_numpy() / bands.n.to_numpy()
axes[0].barh(position, earned, color='#398875', label='Earned')
axes[0].barh(position, precision, left=earned, color='#e9b34b', label='Localization loss')
axes[0].barh(position, skipped, left=earned+precision, color='#c6ccd3', label='Skip loss')
axes[0].set_yticks(position, [f'{row.band} m (n={row.n})' for row in bands.itertuples()])
axes[0].invert_yaxis()
axes[0].set_xlim(0,10)
axes[0].set_xlabel('Points per image (maximum 10)')
axes[0].set_title('Current XoFTR run: 0 wrong in every band')
axes[0].legend(loc='lower center', bbox_to_anchor=(0.5,-0.3), ncol=3)
axes[1].barh([0], [bounds['current_score60']], color='#398875')
axes[1].barh([0], [bands.precision_headroom60.sum()], left=[bounds['current_score60']], color='#e9b34b')
axes[1].barh([0], [bands.skip_headroom60.sum()], left=[bounds['current_score60']+bands.precision_headroom60.sum()], color='#c6ccd3')
axes[1].axvline(50,color='#993c45',linestyle='--', label='Target 50')
axes[1].set_xlim(0,60)
axes[1].set_yticks([])
axes[1].set_ylim(-1,1)
axes[1].set_xlabel('Overall score out of 60')
axes[1].set_title('39.76 earned; 11.57 precision; 8.67 skips')
axes[1].legend()
figure.suptitle('Score accounting, not predicted gains: 332 development frames')
figure.tight_layout()
figure.savefig(results / 'band_loss_budget_codex.png',dpi=160,bbox_inches='tight')
display(figure)
plt.close(figure)

variant_bands = []
for name, frame in frames.items():
    table = grouped_budget(frame, 'band')
    table['variant'] = name
    variant_bands.append(table)
variant_bands = pd.concat(variant_bands,ignore_index=True)
variant_bands.to_csv(results / 'all_variant_bands_codex.csv',index=False)
figure, axis = plt.subplots(figsize=(11,5))
for name in variant_files:
    table = variant_bands[variant_bands.variant.eq(name)]
    axis.plot(band_labels, table.points_per_photo, marker='o',label=name)
axis.set_ylabel('Points per image')
axis.set_xlabel('Height band in metres; observational, not causal')
axis.set_ylim(0,10)
axis.set_title('Completed development runs: 0 wrong for each configuration')
axis.legend(ncol=2)
figure.tight_layout()
figure.savefig(results / 'completed_variants_by_band_codex.png',dpi=160)
display(figure)
plt.close(figure)


In [ ]:
stage_path = source_folder / 'stageeval/tables_highalt_dev332/headings_per_photo_per_variant_official_highalt_dev332_edge_zncc_headings9_scales3_K50_M10.csv'
stage = summarize_stage_diagnostics(stage_path, band_edges, band_labels)
stage.to_csv(results / 'older_stage_diagnostics_codex.csv',index=False)
source_hashes[str(stage_path.relative_to(root))] = file_digest(stage_path)
standard = stage[stage.variant.eq('standard 6 (-5, 0, +5 x 1.0, 1.05)')]
single = stage[stage.variant.eq('plain (heading 0)')]
display(standard[['band','n','target_in_M_fraction','answered_fraction','points_per_photo','wrong_beyond8sigma']].round(3))
print('Older K50/M10 LoFTR report: target means a candidate within 4 scoring sigma.')
print('It does not establish exact retrieval recall in current K50/M5 XoFTR headless runs.')
print('Above1600 m target in M: single',float(single.iloc[-1].target_in_M_fraction),'six',float(standard.iloc[-1].target_in_M_fraction))
print('No current XoFTR raw correspondence/rejection funnel is present in the selected details CSV.')
source_hashes['analysis_block'] = file_digest(folder / 'changed_src/score_budget_codex.py')
(results / 'source_manifest_codex.json').write_text(json.dumps(source_hashes,indent=2))


## How to use this analysis

Read `REPORT_codex.md` for band-wise causes, competing hypotheses and the next tests. Do not choose the best matcher per band from this chart. The truth-picked four-variant result is an optimistic bound of the stored answers, not a selection method. The previously reported locked-test headline is not re-read or used to choose new rules here.

No new pipeline improvement was measured. The strongest new conclusion is an accounting requirement: 50+ needs substantial precision improvement together with recovery; swapping among the currently completed outputs cannot supply it alone.